# Clase 8 — Análisis del modelo

Curso "Deep Learning con Python y PyTorch desde cero" — DIH DATAlife (Hub de
Innovación Digital Europeo)

**Duración estimada:** 100 minutos

## Objetivos de esta clase

- Explicar qué es el sobreajuste (overfitting) y el infra-ajuste
  (underfitting)
- Evaluar un modelo con métricas más allá de la pérdida (precisión, matriz
  de confusión)
- Inspeccionar los pesos y las predicciones de un modelo ya entrenado
- Guardar y cargar un modelo entrenado con `torch.save` y `torch.load`

Trabajamos sobre el dataset MNIST (Modified National Institute of Standards
and Technology, un banco de imágenes de dígitos escritos a mano del 0 al 9),
el mismo que ya usaste en clases anteriores del curso, porque nos permite
centrarnos en el análisis del modelo sin complicar la parte de datos.

## 1. Sobreajuste e infra-ajuste

- **Sobreajuste:** la pérdida (loss, el error del modelo) dentro de muestra
  (in-sample, sobre los datos de entrenamiento) sigue bajando, pero la de
  fuera de muestra (out-of-sample, sobre datos que el modelo no usa para
  ajustar sus pesos) se estanca o sube. El modelo memoriza en vez de
  generalizar
- **Infra-ajuste:** las dos pérdidas se quedan altas. El modelo no tiene
  capacidad o entrenamiento suficiente

Para verlo con nuestros propios ojos, vamos a entrenar un modelo con bastante
más capacidad de la que este problema necesita, sobre un subconjunto
pequeño de datos a propósito. Así conseguimos que sobreajuste en pocas
épocas y podemos observar el fenómeno en directo, sin esperar minutos.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

torch.manual_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Versión de PyTorch:", torch.__version__)
print("Dispositivo:", device)

### Preparamos los datos

Descargamos MNIST con `torchvision` (ya lo hiciste en la clase 2). Para que
el sobreajuste aparezca rápido durante la clase, usamos solo 2000 imágenes
de entrenamiento y 2000 de validación, sacadas ambas del conjunto de
entrenamiento oficial. El conjunto de test oficial (10 000 imágenes) lo
dejamos aparte, sin tocar, para la evaluación final del modelo ya
entrenado.

In [ ]:
transform = transforms.ToTensor()

train_dataset_completo = torchvision.datasets.MNIST(
    root="./datos", train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(
    root="./datos", train=False, download=True, transform=transform)

N_TRAIN = 2000
N_VAL = 2000

indices_train = list(range(N_TRAIN))
indices_val = list(range(N_TRAIN, N_TRAIN + N_VAL))

train_dataset = Subset(train_dataset_completo, indices_train)
val_dataset = Subset(train_dataset_completo, indices_val)

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Ejemplos de entrenamiento: {len(train_dataset)}")
print(f"Ejemplos de validación:   {len(val_dataset)}")
print(f"Ejemplos de test:         {len(test_dataset)}")

### Un modelo con capacidad de sobra

Definimos una red densa (fully connected) con dos capas ocultas de 512
neuronas cada una: mucha más capacidad de la que hace falta para
clasificar dígitos con solo 2000 ejemplos de entrenamiento. Es la receta
perfecta para observar sobreajuste.

In [ ]:
class RedDensa(nn.Module):
    def __init__(self):
        super().__init__()
        self.aplanar = nn.Flatten()
        self.capa1 = nn.Linear(28 * 28, 512)
        self.capa2 = nn.Linear(512, 512)
        self.capa_salida = nn.Linear(512, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.aplanar(x)
        x = self.relu(self.capa1(x))
        x = self.relu(self.capa2(x))
        return self.capa_salida(x)


net = RedDensa().to(device)
funcion_perdida = nn.CrossEntropyLoss()
optimizador = optim.Adam(net.parameters(), lr=1e-3)

print(net)

### Entrenamos registrando las cuatro métricas

En cada época calculamos la pérdida y la precisión dentro de muestra
(sobre `train_loader`) y fuera de muestra (sobre `val_loader`). Para la
parte de validación usamos `torch.no_grad()`, porque no vamos a entrenar
con esos datos y así ahorramos el cálculo de gradientes.

In [ ]:
def evaluar(modelo, data_loader):
    # Calcula la perdida media y la precision de 'modelo' sobre
    # 'data_loader', sin actualizar sus pesos.
    modelo.eval()
    perdida_total = 0.0
    aciertos = 0
    total = 0
    with torch.no_grad():
        for imagenes, etiquetas in data_loader:
            imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
            salidas = modelo(imagenes)
            perdida = funcion_perdida(salidas, etiquetas)

            perdida_total += perdida.item() * etiquetas.size(0)
            predicciones = torch.argmax(salidas, dim=1)
            aciertos += (predicciones == etiquetas).sum().item()
            total += etiquetas.size(0)

    return perdida_total / total, aciertos / total


NUM_EPOCHS = 15
historia = {"perdida_train": [], "perdida_val": [],
            "precision_train": [], "precision_val": []}

for epoca in range(NUM_EPOCHS):
    net.train()
    for imagenes, etiquetas in train_loader:
        imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)

        optimizador.zero_grad()
        salidas = net(imagenes)
        perdida = funcion_perdida(salidas, etiquetas)
        perdida.backward()
        optimizador.step()

    perdida_train, precision_train = evaluar(net, train_loader)
    perdida_val, precision_val = evaluar(net, val_loader)

    historia["perdida_train"].append(perdida_train)
    historia["perdida_val"].append(perdida_val)
    historia["precision_train"].append(precision_train)
    historia["precision_val"].append(precision_val)

    print(f"Época {epoca + 1:2d}/{NUM_EPOCHS} — "
          f"pérdida train: {perdida_train:.3f}, pérdida val: {perdida_val:.3f} — "
          f"precisión train: {precision_train:.2%}, precisión val: {precision_val:.2%}")

In [ ]:
epocas = range(1, NUM_EPOCHS + 1)

plt.figure(figsize=(11, 4))

plt.subplot(1, 2, 1)
plt.plot(epocas, historia["perdida_train"], label="Entrenamiento (in-sample)")
plt.plot(epocas, historia["perdida_val"], "--", label="Validación (out-of-sample)")
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.title("Pérdida por época")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epocas, historia["precision_train"], label="Entrenamiento (in-sample)")
plt.plot(epocas, historia["precision_val"], "--", label="Validación (out-of-sample)")
plt.xlabel("Época")
plt.ylabel("Precisión")
plt.title("Precisión por época")
plt.legend()

plt.tight_layout()
plt.show()

Fíjate en el patrón: la pérdida y la precisión de entrenamiento siguen
mejorando época tras época, mientras que las de validación se estancan (o
incluso empeoran) mucho antes. Esa separación creciente entre las dos curvas
es la firma del sobreajuste: el modelo está memorizando las 2000 imágenes
concretas de entrenamiento en lugar de aprender a reconocer dígitos en
general.

Si en tu ejecución las curvas casi no se separan, prueba a subir
`NUM_EPOCHS` o a reducir `N_TRAIN` en la celda de los datos: cuantos menos
datos y más épocas, más fácil es memorizarlos.

## 2. Evaluar con métricas más allá de la pérdida

La precisión ya nos dice algo más concreto que la pérdida, pero sigue siendo
un único número. Para saber en qué se equivoca el modelo exactamente,
usamos la **matriz de confusión**: una tabla con una fila por cada dígito
real y una columna por cada dígito predicho. En la diagonal quedan los
aciertos; fuera de la diagonal, los errores concretos (por ejemplo, cuántas
veces el modelo confunde un 4 con un 9).

Evaluamos ahora sobre el conjunto de test oficial, que el modelo no ha visto
en ningún momento del entrenamiento.

In [ ]:
NUM_CLASES = 10
matriz_confusion = torch.zeros(NUM_CLASES, NUM_CLASES, dtype=torch.int64)
aciertos = 0
total = 0

net.eval()
with torch.no_grad():
    for imagenes, etiquetas in test_loader:
        imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
        salidas = net(imagenes)
        predicciones = torch.argmax(salidas, dim=1)

        aciertos += (predicciones == etiquetas).sum().item()
        total += etiquetas.size(0)

        for etiqueta_real, etiqueta_predicha in zip(etiquetas.cpu(), predicciones.cpu()):
            matriz_confusion[etiqueta_real, etiqueta_predicha] += 1

precision_test = aciertos / total
print(f"Precisión sobre el conjunto de test: {precision_test:.2%}")

plt.figure(figsize=(6, 5))
plt.imshow(matriz_confusion, cmap="Blues")
plt.colorbar(label="Número de ejemplos")
plt.xlabel("Dígito predicho")
plt.ylabel("Dígito real")
plt.title("Matriz de confusión — conjunto de test")
plt.xticks(range(NUM_CLASES))
plt.yticks(range(NUM_CLASES))
plt.show()

print(matriz_confusion)

Ten en cuenta que este modelo lo entrenamos a propósito con muy pocos datos
para forzar el sobreajuste: su precisión de test será más baja que la de un
modelo entrenado con las 60 000 imágenes completas de MNIST. Lo importante
aquí no es la cifra en sí, sino el proceso: comparar curvas, calcular la
matriz de confusión y, a continuación, mirar ejemplos concretos.

## 3. Inspeccionar predicciones incorrectas

La matriz de confusión te dice qué dígitos se confunden entre sí, pero ver
las imágenes reales de esos errores te ayuda a entender si el error es
razonable (un dígito mal escrito y ambiguo) o si el modelo se equivoca en
algo que a simple vista está claro.

In [ ]:
imagenes_test, etiquetas_test = next(iter(
    DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)))
imagenes_test, etiquetas_test = imagenes_test.to(device), etiquetas_test.to(device)

net.eval()
with torch.no_grad():
    salidas = net(imagenes_test)
    predicciones = torch.argmax(salidas, dim=1)

indices_erroneos = (predicciones != etiquetas_test).nonzero(as_tuple=True)[0]
print(f"Número de predicciones incorrectas sobre el test: {len(indices_erroneos)} de {len(test_dataset)}")

plt.figure(figsize=(10, 2.5))
for i, indice in enumerate(indices_erroneos[:6]):
    indice = indice.item()
    plt.subplot(1, 6, i + 1)
    plt.imshow(imagenes_test[indice].cpu().squeeze(), cmap="gray")
    plt.title(f"Real: {etiquetas_test[indice].item()}\nPred: {predicciones[indice].item()}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Guardar y cargar el modelo

Ya analizamos el modelo a fondo. Ahora vamos a guardar sus pesos con
`torch.save` para no perder el trabajo, y a comprobar que, al recargarlos en
una arquitectura nueva con `torch.load` y `load_state_dict`, el modelo
recuperado predice exactamente igual que el original.

In [ ]:
ruta_modelo = "modelo_clase08.pt"

# Guardamos únicamente los pesos aprendidos (el state_dict), no el objeto
# 'net' completo: es la forma recomendada por la documentación de PyTorch
torch.save(net.state_dict(), ruta_modelo)
print(f"Pesos guardados en '{ruta_modelo}'")

# Reconstruimos la MISMA arquitectura, sin entrenar
modelo_restaurado = RedDensa().to(device)

# Antes de cargar los pesos, sus predicciones deberían ser prácticamente
# aleatorias (alrededor de un 10% de precisión, como elegir al azar entre
# diez dígitos)
_, precision_sin_pesos = evaluar(modelo_restaurado, test_loader)
print(f"Precisión antes de cargar los pesos: {precision_sin_pesos:.2%}")

# Cargamos los pesos guardados dentro de esa arquitectura
modelo_restaurado.load_state_dict(torch.load(ruta_modelo, map_location=device))
modelo_restaurado.eval()

_, precision_con_pesos = evaluar(modelo_restaurado, test_loader)
print(f"Precisión después de cargar los pesos: {precision_con_pesos:.2%}")

# Comprobación más estricta: ¿predicen exactamente lo mismo, ejemplo a
# ejemplo, sobre el mismo lote de imágenes?
with torch.no_grad():
    salida_original = net(imagenes_test)
    salida_restaurada = modelo_restaurado(imagenes_test)

prediccion_original = torch.argmax(salida_original, dim=1)
prediccion_restaurada = torch.argmax(salida_restaurada, dim=1)

print("¿Predicciones idénticas a las del modelo original?",
      torch.equal(prediccion_original, prediccion_restaurada))

## Resumen

- Comparar la pérdida y la precisión dentro de muestra (entrenamiento) y
  fuera de muestra (validación o test) en cada época es la forma más directa
  de detectar sobreajuste (las curvas se separan) o infra-ajuste (las dos se
  quedan altas)
- La precisión de test importa más que la de entrenamiento: es la única
  cifra que se parece a lo que hará el modelo con datos reales
- La matriz de confusión revela en qué categorías concretas se equivoca el
  modelo, algo que un único porcentaje de precisión no muestra
- Mirar ejemplos concretos de predicciones incorrectas ayuda a entender si
  los errores del modelo son razonables o inesperados
- `torch.save(modelo.state_dict(), ruta)` y
  `modelo.load_state_dict(torch.load(ruta))` guardan y recuperan los pesos
  de un modelo; necesitas reconstruir antes la misma arquitectura

Sigue practicando en el notebook
`clase08_practica_analisis_del_modelo.ipynb` y consulta el capítulo 08 del
manual para repasar la teoría con más detalle.